# Demand Prediction for Retail Inventory Optimization

This notebook builds a machine learning model to predict product demand (quantity) using a retail dataset. The workflow includes data preprocessing, feature engineering, model training (Random Forest), evaluation, feature importance analysis, and model saving. Best practices are followed throughout for robust, industry-level results.

In [1]:
import pandas as pd
df = pd.read_csv("walmart.csv", encoding_errors='ignore')
df.head()

,invoice_id,Branch,City,category,unit_price,quantity,date,time,payment_method,rating,profit_margin
0,1,WALM003,San Antonio,Health and beauty,$74.69,7.0,05/01/19,13:08:00,Ewallet,9.1,0.48
1,2,WALM048,Harlingen,Electronic accessories,$15.28,5.0,08/03/19,10:29:00,Cash,9.6,0.48
2,3,WALM067,Haltom City,Home and lifestyle,$46.33,7.0,03/03/19,13:23:00,Credit card,7.4,0.33
3,4,WALM064,Bedford,Health and beauty,$58.22,8.0,27/01/19,20:33:00,Ewallet,8.4,0.33
4,5,WALM013,Irving,Sports and travel,$86.31,7.0,08/02/19,10:37:00,Ewallet,5.3,0.48


In [2]:
df.describe()

,invoice_id,quantity,rating,profit_margin
count,10051.000000,10020.000000,10051.000000,10051.000000
mean,5025.741220,2.353493,5.825659,0.393791
std,2901.174372,1.602658,1.763991,0.090669
min,1.000000,1.000000,3.000000,0.180000
25%,2513.500000,1.000000,4.000000,0.330000
50%,5026.000000,2.000000,6.000000,0.330000
75%,7538.500000,3.000000,7.000000,0.480000
max,10000.000000,10.000000,10.000000,0.570000


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10051 entries, 0 to 10050
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   invoice_id      10051 non-null  int64  
 1   Branch          10051 non-null  object 
 2   City            10051 non-null  object 
 3   category        10051 non-null  object 
 4   unit_price      10020 non-null  object 
 5   quantity        10020 non-null  float64
 6   date            10051 non-null  object 
 7   time            10051 non-null  object 
 8   payment_method  10051 non-null  object 
 9   rating          10051 non-null  float64
 10  profit_margin   10051 non-null  float64
dtypes: float64(3), int64(1), object(7)
memory usage: 863.9+ KB


In [4]:
df.duplicated().sum()


np.int64(51)

In [5]:
df.isnull().sum()

invoice_id         0
Branch             0
City               0
category           0
unit_price        31
quantity          31
date               0
time               0
payment_method     0
rating             0
profit_margin      0
dtype: int64

In [6]:
df.drop_duplicates(inplace=True)

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df.shape

(10000, 11)

dropping all the duplicate values

In [9]:
df.dropna(inplace=True)

In [10]:
df.isnull().sum()

invoice_id        0
Branch            0
City              0
category          0
unit_price        0
quantity          0
date              0
time              0
payment_method    0
rating            0
profit_margin     0
dtype: int64

In [11]:
df.shape

(9969, 11)

In [12]:
df.dtypes

invoice_id          int64
Branch             object
City               object
category           object
unit_price         object
quantity          float64
date               object
time               object
payment_method     object
rating            float64
profit_margin     float64
dtype: object

In [13]:
df['unit_price'] = df['unit_price'].str.replace('$', '').astype(float)
df.head()

,invoice_id,Branch,City,category,unit_price,quantity,date,time,payment_method,rating,profit_margin
0,1,WALM003,San Antonio,Health and beauty,74.69,7.0,05/01/19,13:08:00,Ewallet,9.1,0.48
1,2,WALM048,Harlingen,Electronic accessories,15.28,5.0,08/03/19,10:29:00,Cash,9.6,0.48
2,3,WALM067,Haltom City,Home and lifestyle,46.33,7.0,03/03/19,13:23:00,Credit card,7.4,0.33
3,4,WALM064,Bedford,Health and beauty,58.22,8.0,27/01/19,20:33:00,Ewallet,8.4,0.33
4,5,WALM013,Irving,Sports and travel,86.31,7.0,08/02/19,10:37:00,Ewallet,5.3,0.48


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9969 entries, 0 to 9999
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   invoice_id      9969 non-null   int64  
 1   Branch          9969 non-null   object 
 2   City            9969 non-null   object 
 3   category        9969 non-null   object 
 4   unit_price      9969 non-null   float64
 5   quantity        9969 non-null   float64
 6   date            9969 non-null   object 
 7   time            9969 non-null   object 
 8   payment_method  9969 non-null   object 
 9   rating          9969 non-null   float64
 10  profit_margin   9969 non-null   float64
dtypes: float64(4), int64(1), object(6)
memory usage: 934.6+ KB


In [15]:
df.columns

Index(['invoice_id', 'Branch', 'City', 'category', 'unit_price', 'quantity',
       'date', 'time', 'payment_method', 'rating', 'profit_margin'],
      dtype='object')

In [16]:
df['total'] = df['unit_price'] * df['quantity']

In [17]:
df.head()

,invoice_id,Branch,City,category,unit_price,quantity,date,time,payment_method,rating,profit_margin,total
0,1,WALM003,San Antonio,Health and beauty,74.69,7.0,05/01/19,13:08:00,Ewallet,9.1,0.48,522.83
1,2,WALM048,Harlingen,Electronic accessories,15.28,5.0,08/03/19,10:29:00,Cash,9.6,0.48,76.40
2,3,WALM067,Haltom City,Home and lifestyle,46.33,7.0,03/03/19,13:23:00,Credit card,7.4,0.33,324.31
3,4,WALM064,Bedford,Health and beauty,58.22,8.0,27/01/19,20:33:00,Ewallet,8.4,0.33,465.76
4,5,WALM013,Irving,Sports and travel,86.31,7.0,08/02/19,10:37:00,Ewallet,5.3,0.48,604.17


In [18]:
df["date"] = pd.to_datetime(df["date"])

/var/folders/kq/q43_zd5d1ql9jm6fhw_nvrd00000gn/T/ipykernel_44364/3228555721.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date"] = pd.to_datetime(df["date"])


In [19]:
df["year"] = df["date"].dt.year

In [20]:
df.head()

,invoice_id,Branch,City,category,unit_price,quantity,date,time,payment_method,rating,profit_margin,total,year
0,1,WALM003,San Antonio,Health and beauty,74.69,7.0,2019-05-01,13:08:00,Ewallet,9.1,0.48,522.83,2019
1,2,WALM048,Harlingen,Electronic accessories,15.28,5.0,2019-08-03,10:29:00,Cash,9.6,0.48,76.40,2019
2,3,WALM067,Haltom City,Home and lifestyle,46.33,7.0,2019-03-03,13:23:00,Credit card,7.4,0.33,324.31,2019
3,4,WALM064,Bedford,Health and beauty,58.22,8.0,2019-01-27,20:33:00,Ewallet,8.4,0.33,465.76,2019
4,5,WALM013,Irving,Sports and travel,86.31,7.0,2019-08-02,10:37:00,Ewallet,5.3,0.48,604.17,2019


In [21]:
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day

In [22]:
df.head()

,invoice_id,Branch,City,category,unit_price,quantity,date,time,payment_method,rating,profit_margin,total,year,month,day
0,1,WALM003,San Antonio,Health and beauty,74.69,7.0,2019-05-01,13:08:00,Ewallet,9.1,0.48,522.83,2019,5,1
1,2,WALM048,Harlingen,Electronic accessories,15.28,5.0,2019-08-03,10:29:00,Cash,9.6,0.48,76.40,2019,8,3
2,3,WALM067,Haltom City,Home and lifestyle,46.33,7.0,2019-03-03,13:23:00,Credit card,7.4,0.33,324.31,2019,3,3
3,4,WALM064,Bedford,Health and beauty,58.22,8.0,2019-01-27,20:33:00,Ewallet,8.4,0.33,465.76,2019,1,27
4,5,WALM013,Irving,Sports and travel,86.31,7.0,2019-08-02,10:37:00,Ewallet,5.3,0.48,604.17,2019,8,2


In [23]:
df.columns

Index(['invoice_id', 'Branch', 'City', 'category', 'unit_price', 'quantity',
       'date', 'time', 'payment_method', 'rating', 'profit_margin', 'total',
       'year', 'month', 'day'],
      dtype='object')

In [24]:
df["City"].value_counts().sum()

np.int64(9969)

In [25]:
df.groupby("City")["total"].sum().sort_values(ascending=False)

City
Weslaco         46351.79
Waxahachie      40703.33
Plano           25688.34
San Antonio     24950.56
Port Arthur     24524.37
                  ...   
Longview         6769.33
Pearland         6572.91
Irving           6237.11
Lewisville       5568.84
Lake Jackson     5038.90
Name: total, Length: 98, dtype: float64

In [26]:
df.groupby("month")["total"].sum().sort_values(ascending=False)

month
12    185343.48
11    183972.35
1     144028.54
3     115658.00
2     109313.66
9      86269.70
8      83173.74
10     81188.43
6      58313.50
5      57337.23
7      55140.06
4      49987.69
Name: total, dtype: float64

most selling months : 12 > 11 > 1 > 3 > 2 > 9 > 8 > 10 > 6 > 5 > 7 > 4

In [27]:
df.groupby("City")["total"].sum().sort_values(ascending=False).head(10)

City
Weslaco        46351.79
Waxahachie     40703.33
Plano          25688.34
San Antonio    24950.56
Port Arthur    24524.37
Richardson     24460.60
Rockwall       24077.70
Round Rock     23327.34
Schertz        23095.43
San Marcos     22124.51
Name: total, dtype: float64

In [28]:
df.groupby("category")["total"].sum().sort_values(ascending=False)

category
Fashion accessories       489480.90
Home and lifestyle        489250.06
Electronic accessories     78175.03
Food and beverages         53471.28
Sports and travel          52497.93
Health and beauty          46851.18
Name: total, dtype: float64

In [29]:
df.dtypes

invoice_id                 int64
Branch                    object
City                      object
category                  object
unit_price               float64
quantity                 float64
date              datetime64[ns]
time                      object
payment_method            object
rating                   float64
profit_margin            float64
total                    float64
year                       int32
month                      int32
day                        int32
dtype: object

In [30]:
df["quantity"]=df["quantity"].astype(int)

In [31]:
df["hour"] = pd.to_datetime(df["time"], format="%H:%M:%S").dt.hour

In [32]:
df.groupby("hour")["total"].sum()

hour
6      30839.00
7      35325.00
8      29591.00
9      34295.00
10     60756.22
11     60700.79
12     57820.65
13     66288.74
14     61472.38
15    142016.77
16    134918.07
17    116301.16
18    113072.80
19    128581.06
20    109066.74
21     15182.00
22     13236.00
23       263.00
Name: total, dtype: float64

In [33]:
df.groupby("payment_method")["total"].sum().sort_values(ascending=False)

payment_method
Credit card    488821.02
Ewallet        457316.07
Cash           263589.29
Name: total, dtype: float64

In [34]:
df.groupby("payment_method")["total"].mean()

payment_method
Cash           143.880617
Credit card    114.854563
Ewallet        117.834597
Name: total, dtype: float64

In [35]:
invoice_summary = df.groupby("invoice_id").agg({
    "quantity": "sum",
    "total": "sum",
    "category" : "sum"
})

invoice_summary["avg_price_per_item"] = (
    invoice_summary["total"] / invoice_summary["quantity"]
)

invoice_summary.head()

,quantity,total,category,avg_price_per_item
invoice_id,,,,
1,7,522.83,Health and beauty,74.69
2,5,76.40,Electronic accessories,15.28
3,7,324.31,Home and lifestyle,46.33
4,8,465.76,Health and beauty,58.22
5,7,604.17,Sports and travel,86.31


In [36]:
df.columns

Index(['invoice_id', 'Branch', 'City', 'category', 'unit_price', 'quantity',
       'date', 'time', 'payment_method', 'rating', 'profit_margin', 'total',
       'year', 'month', 'day', 'hour'],
      dtype='object')

In [37]:
df.dtypes

invoice_id                 int64
Branch                    object
City                      object
category                  object
unit_price               float64
quantity                   int64
date              datetime64[ns]
time                      object
payment_method            object
rating                   float64
profit_margin            float64
total                    float64
year                       int32
month                      int32
day                        int32
hour                       int32
dtype: object

TOP PERFORMING BRANCHES

In [38]:
df.groupby("Branch")["total"].sum().sort_values(ascending=False).head(10)

Branch
WALM009    25688.34
WALM074    25555.42
WALM003    24950.56
WALM058    24524.37
WALM030    24460.60
WALM069    24077.70
WALM029    23327.34
WALM084    23095.43
WALM075    22124.51
WALM089    21267.06
Name: total, dtype: float64

average quantity per purchase

In [39]:
df[["unit_price","quantity"]].corr()

,unit_price,quantity
unit_price,1.000000,0.061789
quantity,0.061789,1.000000


In [40]:
df["total"] /df["quantity"].mean()

0       221.951721
1        32.433318
2       137.676038
3       197.724373
4       256.482167
           ...    
9995     47.121705
9996     49.244304
9997     66.225099
9998     67.074139
9999     78.960695
Name: total, Length: 9969, dtype: float64

payment method is used for expensive purchases

In [41]:
df.sort_values(by="total", ascending=False).head(10)["payment_method"].value_counts()

payment_method
Credit card    4
Ewallet        4
Cash           2
Name: count, dtype: int64

In [42]:
df.groupby("category")["quantity"].sum().sort_values(ascending=False)

category
Fashion accessories       9653
Home and lifestyle        9610
Electronic accessories    1494
Food and beverages         952
Sports and travel          920
Health and beauty          854
Name: quantity, dtype: int64

In [43]:
df.groupby("category")["total"].sum().sort_values(ascending=False)

category
Fashion accessories       489480.90
Home and lifestyle        489250.06
Electronic accessories     78175.03
Food and beverages         53471.28
Sports and travel          52497.93
Health and beauty          46851.18
Name: total, dtype: float64

In [44]:
df.groupby("category")["profit_margin"].sum().sort_values(ascending=False)

category
Home and lifestyle        1783.50
Fashion accessories       1783.05
Electronic accessories     164.73
Food and beverages          69.66
Sports and travel           63.45
Health and beauty           60.84
Name: profit_margin, dtype: float64

In [45]:
df.groupby("category")["rating"].sum().sort_values(ascending=False)

category
Fashion accessories       26239.2
Home and lifestyle        25941.0
Electronic accessories     2477.2
Food and beverages         1237.7
Sports and travel          1148.1
Health and beauty          1064.5
Name: rating, dtype: float64

In [46]:
df.groupby("month")["total"].sum().sort_values(ascending=False)

month
12    185343.48
11    183972.35
1     144028.54
3     115658.00
2     109313.66
9      86269.70
8      83173.74
10     81188.43
6      58313.50
5      57337.23
7      55140.06
4      49987.69
Name: total, dtype: float64

In [47]:
df["month_name"] = pd.to_datetime(df["month"], format="%m").dt.month_name()

In [48]:
df.head()

,invoice_id,Branch,City,category,unit_price,quantity,date,time,payment_method,rating,profit_margin,total,year,month,day,hour,month_name
0,1,WALM003,San Antonio,Health and beauty,74.69,7,2019-05-01,13:08:00,Ewallet,9.1,0.48,522.83,2019,5,1,13,May
1,2,WALM048,Harlingen,Electronic accessories,15.28,5,2019-08-03,10:29:00,Cash,9.6,0.48,76.40,2019,8,3,10,August
2,3,WALM067,Haltom City,Home and lifestyle,46.33,7,2019-03-03,13:23:00,Credit card,7.4,0.33,324.31,2019,3,3,13,March
3,4,WALM064,Bedford,Health and beauty,58.22,8,2019-01-27,20:33:00,Ewallet,8.4,0.33,465.76,2019,1,27,20,January
4,5,WALM013,Irving,Sports and travel,86.31,7,2019-08-02,10:37:00,Ewallet,5.3,0.48,604.17,2019,8,2,10,August


In [49]:
df.groupby("month_name")["total"].sum().sort_values(ascending=False)

month_name
December     185343.48
November     183972.35
January      144028.54
March        115658.00
February     109313.66
September     86269.70
August        83173.74
October       81188.43
June          58313.50
May           57337.23
July          55140.06
April         49987.69
Name: total, dtype: float64

In [50]:
df.groupby("time")["total"].sum().sort_values(ascending=False)

time
17:16:00    4991.69
15:42:00    4713.57
16:19:00    4497.68
19:48:00    4276.31
15:01:00    3932.06
             ...   
14:07:00      40.00
6:27:00       38.00
21:09:00      32.00
22:41:00      29.00
11:13:00      22.00
Name: total, Length: 1001, dtype: float64

In [51]:
df["quantity"].corr(df["total"])

np.float64(0.8001564293555931)

expensive items do not affect the rating

In [52]:
df["unit_price"].corr(df["rating"])

np.float64(0.0194071342875651)

In [53]:
df[["unit_price","quantity"]].describe()

,unit_price,quantity
count,9969.000000,9969.000000
mean,50.622142,2.355602
std,21.203766,1.605455
min,10.080000,1.000000
25%,32.000000,1.000000
50%,51.000000,2.000000
75%,69.000000,3.000000
max,99.960000,10.000000


In [54]:
Q1 = df["total"].quantile(0.25)
Q3 = df["total"].quantile(0.75)
IQR = Q3 - Q1

df[(df["total"] < Q1 - 1.5*IQR) | (df["total"] > Q3 + 1.5*IQR)]

,invoice_id,Branch,City,category,unit_price,quantity,date,time,payment_method,rating,profit_margin,total,year,month,day,hour,month_name
0,1,WALM003,San Antonio,Health and beauty,74.69,7,2019-05-01,13:08:00,Ewallet,9.1,0.48,522.83,2019,5,1,13,May
2,3,WALM067,Haltom City,Home and lifestyle,46.33,7,2019-03-03,13:23:00,Credit card,7.4,0.33,324.31,2019,3,3,13,March
3,4,WALM064,Bedford,Health and beauty,58.22,8,2019-01-27,20:33:00,Ewallet,8.4,0.33,465.76,2019,1,27,20,January
4,5,WALM013,Irving,Sports and travel,86.31,7,2019-08-02,10:37:00,Ewallet,5.3,0.48,604.17,2019,8,2,10,August
5,6,WALM026,Denton,Electronic accessories,85.39,7,2019-03-25,18:30:00,Ewallet,4.1,0.48,597.73,2019,3,25,18,March
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
988,989,WALM015,Grand Prairie,Electronic accessories,82.34,10,2019-03-29,19:12:00,Ewallet,4.3,0.48,823.40,2019,3,29,19,March
989,990,WALM095,Big Spring,Health and beauty,75.37,8,2019-01-28,15:46:00,Credit card,8.4,0.33,602.96,2019,1,28,15,January
991,992,WALM001,Houston,Sports and travel,76.60,10,2019-01-24,18:10:00,Ewallet,6.0,0.36,766.00,2019,1,24,18,January
996,997,WALM005,Fort Worth,Home and lifestyle,97.38,10,2019-02-03,17:16:00,Ewallet,4.4,0.48,973.80,2019,2,3,17,February


In [55]:
df["unit_price"].describe()

count    9969.000000
mean       50.622142
std        21.203766
min        10.080000
25%        32.000000
50%        51.000000
75%        69.000000
max        99.960000
Name: unit_price, dtype: float64

In [56]:
df["total"].describe()

count    9969.000000
mean      121.348819
std       112.678040
min        10.170000
25%        54.000000
50%        88.000000
75%       156.000000
max       993.000000
Name: total, dtype: float64

In [57]:
df.columns

Index(['invoice_id', 'Branch', 'City', 'category', 'unit_price', 'quantity',
       'date', 'time', 'payment_method', 'rating', 'profit_margin', 'total',
       'year', 'month', 'day', 'hour', 'month_name'],
      dtype='object')

## 1. Data Preprocessing

- Convert date columns to datetime if needed
- Use extracted features: month, day, hour
- Drop irrelevant or leakage-prone columns: invoice_id, date, time, month_name, year, total
- One-hot encode categorical variables: Branch, City, category, payment_method
- Ensure clean, ready-to-model data

In [ ]:
# Data Preprocessing
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import pickle

# Reload data to ensure clean state
raw_df = pd.read_csv("walmart.csv", encoding_errors='ignore')

# Convert 'date' to datetime if not already
df = raw_df.copy()
df['date'] = pd.to_datetime(df['date'])

# Drop irrelevant/leakage columns
cols_to_drop = ['invoice_id', 'date', 'time', 'month_name', 'year', 'total']
df = df.drop(columns=cols_to_drop)

# One-hot encode categorical variables
categorical_cols = ['Branch', 'City', 'category', 'payment_method']
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

# Check processed data
df.head()

## 2. Feature Selection

- Use only features available before the sale (no future or target-derived columns)
- Avoid data leakage by excluding 'total' and other post-sale info
- Features: unit_price, profit_margin, rating, month, day, hour, and one-hot encoded categorical variables

In [ ]:
# Feature Selection
X = df.drop('quantity', axis=1)
y = df['quantity']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

## 3. Model Training: Random Forest Regressor

- Random Forest is robust for tabular data and handles non-linearity well
- Reasonable hyperparameters: n_estimators=100, max_depth=10 (can be tuned)
- Fit model on training data

In [ ]:
# Model Training
rf = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

## 4. Model Evaluation

- Evaluate using Mean Absolute Error (MAE) and R2 Score
- Lower MAE and higher R2 indicate better performance

In [ ]:
# Model Evaluation
y_pred = rf.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"R2 Score: {r2:.2f}")

## 5. Feature Importance

- Identify top features influencing demand
- Helps in business understanding and further feature engineering

In [ ]:
# Feature Importance
import matplotlib.pyplot as plt
import numpy as np

importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]
features = X.columns

plt.figure(figsize=(10,6))
plt.title("Top 10 Feature Importances")
plt.bar(range(10), importances[indices][:10], align="center")
plt.xticks(range(10), features[indices][:10], rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 6. Model Saving

- Save the trained model using pickle for future inference or deployment

In [ ]:
# Save the trained model
with open("demand_rf_model.pkl", "wb") as f:
    pickle.dump(rf, f)
print("Model saved as demand_rf_model.pkl")

## 7. Next Steps & Best Practices

**Ways to Improve Model Accuracy:**
- Hyperparameter tuning (GridSearchCV, RandomizedSearchCV)
- Feature engineering (e.g., lag features, rolling averages, holiday flags)
- Try advanced models (XGBoost, LightGBM)
- Address class imbalance if present
- Cross-validation for robust evaluation

**Extending for Real-World Deployment:**
- Automate data pipeline (ETL)
- Monitor model drift and retrain periodically
- Deploy as a REST API (Flask, FastAPI)
- Integrate with inventory management systems
- Add explainability (SHAP, LIME)
- Ensure data privacy and security

---

**This notebook provides a robust foundation for AI-driven inventory and sales optimization in retail.**